<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/STEP_2_FEATURE_ENGINEERING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 2: FEATURE ENGINEERING**

In [ ]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

# Date and Time Features

In [ ]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)

# Create a Product-Region Matrix and Network

In [ ]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

print(product_region.head(20))

In [ ]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [ ]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [ ]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())

# Define Multi-Region Stock

In [ ]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [ ]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))

# Merge the network

In [ ]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [ ]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))

 # Find candidate shipping regions

In [ ]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Shipping Engineering

In [ ]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [ ]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [ ]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [ ]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [ ]:
stocking = pd.read_csv(
    "product_stocking_scores.csv"
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [ ]:
df['shipping_mode_speed'] = df['Shipping Mode'].map({
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
})

df['action_id'] = df['Shipping Mode'].map(shipping_mode_speed)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [ ]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# RL State Features

In [ ]:
state_features = [
    'Product Category Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_weekday'
]

# Reward Features

In [ ]:
df['reward'] = (
    df['Order Profit Per Order']
    + 100 * df['on_time_delivery']
    - 20 * df['late_delivery']
)
LATE_PENALTY = 10

df['reward'] = (
    df['Order Profit Per Order']
    - df['Mode_Cost']
    - LATE_PENALTY * df['Late_delivery_risk']
)

# Save Outputs

In [ ]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)


print("Files created successfully")